---
## Step 1 — Library Imports & Reproducibility Seeds


In [1]:
import numpy as np
import pandas as pd
import itertools
import re
import os
import random
import time

from Bio import SeqIO
from Bio.SeqUtils import gc_fraction
from tqdm import tqdm
from sklearn.utils import shuffle

# Reproducibility
random.seed(12)
np.random.seed(12)

print("All libraries loaded.")
print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")


All libraries loaded.
NumPy  : 2.3.5
Pandas : 2.3.3


---
## Step 2 — Extract VISTA Enhancer Coordinates (chr1)
**What we do:** Parse the VISTA enhancer FASTA file to extract genomic
coordinates, then filter to chromosome 1 and write a BED file.  
**Why it matters:** The VISTA database delivers sequences in FASTA format with
coordinates encoded in the header (e.g. `hg19|chr1:1000-1640|...`).
We need a BED file (chromosome, start, end) so that `bedtools getfasta` can
later retrieve sequences from the reference genome at standardised window sizes.  
**Input:** `hg19.VISTA.enhancers.fa`  
**Output:** `hg19.VISTA.enhancers.chr1.bed`


In [2]:
in_fa = "hg19.VISTA.enhancers.fa"
beds  = []

for record in tqdm(SeqIO.parse(in_fa, "fasta"), desc="Parsing VISTA headers"):
    rec = re.split(r"[|:-]", record.name)   # split on |  :  -
    beds.append(rec[1:])                     # [chrom, start, end, tissue, ...]

count = 0
with open("hg19.VISTA.enhancers.chr1.bed", "w") as bed:
    for reg in beds:
        if re.match(r"^chr1$", reg[0]):
            bed.write(f"{reg[0]}\t{reg[1]}\t{reg[2]}\n")
            count += 1

print(f"Chr1 enhancers written: {count}")


Parsing VISTA headers: 0it [00:00, ?it/s]

Parsing VISTA headers: 3889it [00:00, 16624.67it/s]

Chr1 enhancers written: 0


### 2b — Retrieve chr1 Enhancer Sequences from the Reference Genome
**Run in terminal / Colab `!` cell:**
```bash
bedtools getfasta -fi hg19.fa \
                  -bed hg19.VISTA.enhancers.chr1.bed \
                  -fo  hg19.VISTA.enhancers.chr1.fa
```
`bedtools getfasta` looks up each BED coordinate in the reference genome FASTA
and writes the corresponding DNA sequence to a new FASTA file.


---
## Step 3 — Compute Sequence Statistics
**What we do:** Calculate length, GC content, and percentage of ambiguous
bases (N) for every enhancer sequence.  
**Why it matters:** These statistics reveal the GC distribution of real
enhancers — information we use in Step 7 to build a matched negative set.
Knowing the mean and standard deviation of GC content lets us discard
background sequences that differ too much from enhancers, preventing the
classifier from learning GC bias instead of true enhancer patterns.  
**Input:** `hg19.VISTA.enhancers.chr1.fa`


In [ ]:
def fa_stats(in_fa):
    """Return dict of per-record Length, GC%, and %N for a FASTA file."""
    stats = {"Lengths": {}, "GC content (%)": {}, "Percentage of Ns": {}}
    for i, record in tqdm(enumerate(SeqIO.parse(in_fa, "fasta")), desc="fa_stats"):
        seq = str(record.seq).upper()
        key = f"seq_{i}"
        stats["Lengths"][key]          = len(seq)
        stats["GC content (%)"][key]   = gc_fraction(record.seq) * 100
        stats["Percentage of Ns"][key] = (seq.count("N") / len(seq)) * 100
    return stats

# Chr1 enhancer stats
enh1_stats = fa_stats("hg19.VISTA.enhancers.chr1.fa")
enh1_df    = pd.DataFrame.from_dict(enh1_stats)
print("=== Chr1 Enhancer Statistics ===")
print(enh1_df.describe().round(3))

# Full VISTA set stats (used for GC bounds in Step 7)
enh_stats = fa_stats("hg19.VISTA.enhancers.fa")
enh_df    = pd.DataFrame.from_dict(enh_stats)
print("\n=== Full VISTA Enhancer Set Statistics ===")
print(enh_df.describe().round(3))

# Save GC bounds for use in Step 7
mean    = enh1_df["GC content (%)"].mean()
std_dev = enh1_df["GC content (%)"].std()
print(f"\nGC filter bounds: {mean:.4f} +/- {std_dev:.4f}")
print(f"  Lower: {mean - std_dev:.4f}  |  Upper: {mean + std_dev:.4f}")


fa_stats: 336it [00:00, 5397.43it/s]


=== Chr1 Enhancer Statistics ===
         Lengths  GC content (%)  Percentage of Ns
count    336.000         336.000             336.0
mean    2191.310          45.692               0.0
std     1656.212           7.904               0.0
min       94.000          26.165               0.0
25%     1133.000          39.628               0.0
50%     1694.500          45.538               0.0
75%     2941.500          51.501               0.0
max    13850.000          69.477               0.0


fa_stats: 3889it [00:01, 3636.73it/s]


=== Full VISTA Enhancer Set Statistics ===
         Lengths  GC content (%)  Percentage of Ns
count   3889.000        3889.000            3889.0
mean    2225.636          43.728               0.0
std     1664.134           7.764               0.0
min       42.000          22.601               0.0
25%     1143.000          37.886               0.0
50%     1729.000          42.369               0.0
75%     2951.000          48.848               0.0
max    19876.000          76.391               0.0

GC filter bounds: 45.6916 +/- 7.9036
  Lower: 37.7880  |  Upper: 53.5951


---
## Step 4 — Standardise Positive Sequences to 200 bp Windows
**What we do:** Convert variable-length enhancer regions into fixed 200 bp
windows using centre-expansion (short regions) or sliding windows (long regions).  
**Why it matters:** Machine learning models require fixed-length inputs. 200 bp
is the standard window size in regulatory genomics — it approximates one
nucleosome footprint and captures enough sequence context for motif analysis.  
**Input:** `hg19.VISTA.enhancers.chr1.bed`  
**Output:** `hg19.VISTA.enhancers.chr1.200bp.bed`


In [4]:
# Replicating grep functionality in Python
input_file = "hg19.vistaEnhancers.bed"
output_file = "hg19.VISTA.enhancers.chr1.bed"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        if line.startswith("chr1"):
            outfile.write(line)

print(f"Finished writing chr1 enhancers to {output_file}")


Finished writing chr1 enhancers to hg19.VISTA.enhancers.chr1.bed


In [5]:
def bed_extract(in_bed, outfile, window=200, step=1, label="enhancer"):
    """
    Produce fixed-size windows from a BED file.
    - Regions shorter than window: centre-expand.
    - Regions >= window: slide across with given step.
    """
    df = pd.read_csv(in_bed, sep="\t", header=None)
    with open(outfile, "w") as out:
        for row in tqdm(df.itertuples(), total=len(df), desc="bed_extract"):
            chrom, start, stop = row[1], row[2], row[3]
            if stop - start < window:
                mid = start + (stop - start) // 2
                out.write(f"{chrom}\t{mid - window//2}\t{mid + window//2}\t{label}\n")
            else:
                for k in range(0, stop - start + 1, step):
                    if start + k + window <= stop:
                        out.write(f"{chrom}\t{start+k}\t{start+k+window}\t{label}\n")
    print(f"Saved → {outfile}")

bed_extract("hg19.VISTA.enhancers.chr1.bed",
            "hg19.VISTA.enhancers.chr1.200bp.bed",
            window=200, step=1, label="enhancer")


bed_extract:   0%|          | 0/1706 [00:00<?, ?it/s]

bed_extract: 100%|██████████| 1706/1706 [00:08<00:00, 206.88it/s]


Saved → hg19.VISTA.enhancers.chr1.200bp.bed


---
## Step 5 — Extract Gene Annotations & Promoter Regions
**What we do:** Parse the Ensembl GTF file to identify all protein-coding genes
on chr1, then define promoter regions as ±1000 bp around each transcription
start site (TSS), respecting strand orientation.  
**Why it matters:** Promoters are the primary regulatory regions and must be
**excluded** from the negative set. If we accidentally included real promoters
as "non-enhancers", the model would learn conflicting signals.  
**Input:** `Homo_sapiens.GRCh37.87.chr.gtf`  
**Output:** `Homo_sapiens.GRCh37.87.protein-coding.chr1.promoter_regions.bed`


In [7]:
pwd

'e:\\3_bio_project_ml'

In [9]:
import pandas as pd

genes = pd.read_csv(
    r"e:\3_bio_project_ml\Homo_sapiens.GRCh37.87.chr.gtf",
    header=None,
    sep="\t",
    comment="#"
)


# ── Find protein-coding gene rows ────────────────────────────────────────────
protein_coding_idx = []
for row in tqdm(genes.itertuples(), total=len(genes), desc="Scanning GTF"):
    if re.match(r"gene", str(row[3])) and "protein_coding" in str(row[9]):
        protein_coding_idx.append(row[0])

protein_df = genes.iloc[protein_coding_idx].reset_index(drop=True)
print(f"Protein-coding genes found: {len(protein_coding_idx)}")

# ── Write promoter regions (±1000 bp from TSS, strand-aware) ─────────────────
with open(r"e:\3_bio_project_ml\Homo_sapiens.GRCh37.87.protein-coding.chr1.promoter_regions.bed", "w") as out:
    for row in tqdm(protein_df.itertuples(), total=len(protein_df), desc="Promoters"):
        if str(row[1]) != "1":          # chr1 only
            continue
        if re.match(r"\+", str(row[7])):
            out.write(f"chr{row[1]}\t{row[4]-1000}\t{row[4]+1000}\tpromoter\n")
        elif re.match(r"-", str(row[7])):
            out.write(f"chr{row[1]}\t{row[5]-1000}\t{row[5]+1000}\tpromoter\n")

print("Promoter BED saved.")


C:\Users\IT\AppData\Local\Temp\ipykernel_9148\4034539005.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  genes = pd.read_csv(
Scanning GTF: 100%|██████████| 2612761/2612761 [00:08<00:00, 307341.16it/s]


Protein-coding genes found: 20327


Promoters: 100%|██████████| 20327/20327 [00:00<00:00, 356881.73it/s]

Promoter BED saved.


---
## Step 6 — Filter ENCODE Blacklist Regions (chr1)
**What we do:** Extract chromosome 1 entries from the ENCODE blacklist and
save them as a separate BED file.  
**Why it matters:** Blacklisted regions (centromeric repeats, satellite DNA,
multi-mapping loci) produce artefactual signals in sequencing experiments.
Including them in the negative set would teach the model to recognise
sequencing artefacts, not genuine non-enhancer sequence biology.  
**Input:** `hg19-blacklist.bed`  
**Output:** `hg19-blacklist.chr1.bed`


In [10]:
blacklist = pd.read_csv( r"e:\3_bio_project_ml\hg19-blacklist.bed", sep="\t", header=None)

with open(r"e:\3_bio_project_ml\hg19-blacklist.chr1.bed", "w") as out:
    for row in tqdm(blacklist.itertuples(), total=len(blacklist), desc="Blacklist"):
        if row[1] == "chr1":
            out.write(f"{row[1]}\t{row[2]}\t{row[3]}\t{row[4]}\n")

print("Chr1 blacklist saved.")


Blacklist: 100%|██████████| 411/411 [00:00<00:00, 251731.74it/s]

Chr1 blacklist saved.


### 6b — Build Annotated Region List & Complement (Terminal Commands)
```bash
# Merge all annotated regions into one sorted BED
cat hg19-blacklist.chr1.bed \
    Homo_sapiens.GRCh37.87.chr1.bed \
    hg19.VISTA.enhancers.chr1.200bp.bed \
    Homo_sapiens.GRCh37.87.protein-coding.chr1.promoter_regions.bed \
  | sort -k1,1 -k2,2n > hg19.annotated.regions.chr1.bed

# Find everything NOT annotated → candidate negative regions
fasize -detailed hg19.fa | sort -k1,1 -k2,2n | \
  bedtools complement -L \
    -i hg19.annotated.regions.chr1.bed \
    -g - > hg19.unannotated.regions.chr1.bed
```
The complement operation returns all genomic intervals that are not covered by
any annotated region, giving us clean genomic background for negative examples.


---
## Step 7 — Generate & GC-Balance the Negative Set
**What we do:** Slide 200 bp windows across unannotated background regions to
create candidate negative sequences, retrieve their FASTA sequences, then keep
only those whose GC content falls within mean ± 1 SD of the positive set.  
**Why it matters:** Enhancers cluster in GC-rich open chromatin. Without GC
matching, a naive classifier would simply learn GC content, not true enhancer
biology. Matching the GC distribution forces the model to learn sequence-level
motifs instead.  
**Input:** `hg19.unannotated.regions.chr1.bed` + reference genome FASTA  
**Output:** `hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.{split}.clean.fa`


In [11]:
def neg_bed_extract(in_bed, outfile, window=200, step=1, label="non_enhancer"):
    """Slide fixed windows across unannotated regions (skip short regions)."""
    df = pd.read_csv(in_bed, sep="\t", header=None)
    with open(outfile, "w") as out:
        for row in tqdm(df.itertuples(), total=len(df), desc="neg_bed_extract"):
            chrom, start, stop = row[1], row[2], row[3]
            if stop - start < window:
                continue
            for k in range(0, stop - start + 1, step):
                if start + k + window <= stop:
                    out.write(f"{chrom}\t{start+k}\t{start+k+window}\t{label}\n")
    print(f"Saved → {outfile}")


**After running `neg_bed_extract`, retrieve sequences with bedtools:**
```bash
for i in hg19.VISTA.non_enhancers.chr1.200bp.*.bed; do
    bedtools getfasta -fi hg19.fa -bed ${i} \
        -fo $(basename -s .bed ${i}).fa
done
```


In [12]:
# GC-balance filter — keep negatives with GC within mean ± std_dev of positives
# mean and std_dev were computed in Step 3
for split in ["train", "val", "test"]:
    # Correct way: raw f-string
    in_fa  = rf"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.200bp.{split}.fa"
    out_fa = rf"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.{split}.clean.fa"

    kept = 0
    with open(out_fa, "w") as fa_out:
        for record in tqdm(SeqIO.parse(in_fa, "fasta"), desc=f"GC-balance [{split}]"):
            if re.match(r"chr[A-Za-z]", record.name):   # skip non-standard chroms
                continue
            gc = gc_fraction(record.seq) * 100
            if (mean - std_dev) <= gc <= (mean + std_dev):
                SeqIO.write(record, fa_out, "fasta")
                kept += 1
    print(f"  [{split}] kept {kept} GC-balanced negatives → {out_fa}")


GC-balance [train]: 856147it [00:47, 17848.23it/s]


  [train] kept 405079 GC-balanced negatives → e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.train.clean.fa


GC-balance [val]: 245275it [00:11, 21165.04it/s]


  [val] kept 116274 GC-balanced negatives → e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.val.clean.fa


GC-balance [test]: 123025it [00:04, 26101.67it/s]


  [test] kept 58344 GC-balanced negatives → e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.test.clean.fa


---
## Step 8 — Train / Validation / Test Split (70 / 20 / 10)
**What we do:** Shuffle and split both positive and negative BED files into
training (70%), validation (20%), and test (10%) sets, then retrieve the
corresponding FASTA sequences.  
**Why it matters:** A held-out test set that the model never sees during
training gives an honest, unbiased estimate of generalisation performance.
The validation set is used for hyperparameter tuning without contaminating
the test set.  
**Input:** `hg19.VISTA.enhancers.chr1.200bp.bed` (positive) +
           `hg19.VISTA.non_enhancers.chr1.200bp.bed` (negative)  
**Output:** `*.train.bed`, `*.val.bed`, `*.test.bed` → then FASTA via bedtools


In [13]:
def split_bed(in_bed, out_prefix, random_state=12):
    """Shuffle and split a BED file 70/20/10 → train/val/test BED files."""
    df    = pd.read_csv(in_bed, sep="\t", header=None).drop_duplicates()
    idx   = shuffle(np.arange(len(df)), random_state=random_state)
    n     = len(idx)
    train_idx = idx[:int(n * 0.7)]
    val_idx   = idx[int(n * 0.7):int(n * 0.9)]
    test_idx  = idx[int(n * 0.9):]

    for split_name, split_idx in [("train", train_idx),
                                   ("val",   val_idx),
                                   ("test",  test_idx)]:
        out_path = f"{out_prefix}.{split_name}.bed"
        subset   = df.iloc[split_idx]
        with open(out_path, "w") as out:
            for row in subset.itertuples():
                out.write(f"{row[1]}\t{row[2]}\t{row[3]}\n")
        print(f"  [{split_name}] {len(split_idx)} regions → {out_path}")

# ── Split positive set ────────────────────────────────────────────────────────
print("Splitting positive (enhancer) regions:")
split_bed(r"e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.bed",
          r"e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp")

# ── Split negative set ────────────────────────────────────────────────────────
print("\nSplitting negative (non-enhancer) regions:")
split_bed(r"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.200bp.bed",
          r"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.200bp")


Splitting positive (enhancer) regions:
  [train] 171779 regions → e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.train.bed
  [val] 49080 regions → e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.val.bed
  [test] 24540 regions → e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.test.bed

Splitting negative (non-enhancer) regions:
  [train] 4501199 regions → e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.200bp.train.bed
  [val] 1286057 regions → e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.200bp.val.bed
  [test] 643029 regions → e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.200bp.test.bed


**Retrieve split FASTA sequences (run in terminal):**
```bash
# Positive splits
for i in hg19.VISTA.enhancers.chr1.200bp.*.bed; do
    bedtools getfasta -fi hg19.fa -bed ${i} -fo $(basename -s .bed ${i}).fa
done
# Negative splits
for i in hg19.VISTA.non_enhancers.chr1.200bp.*.bed; do
    bedtools getfasta -fi hg19.fa -bed ${i} -fo $(basename -s .bed ${i}).fa
done
```


---
## Step 9 — K-mer Feature Extraction (k = 1, 2, 3 → 84 features)
**What we do:** Convert each DNA sequence into a 84-dimensional numerical
vector of normalised k-mer frequencies (4 mono-nucleotides + 16 di-nucleotides
+ 64 tri-nucleotides).  
**Why it matters:** Machine learning models cannot read raw DNA letters.
K-mer frequencies encode the sequence's nucleotide composition and short
sequence context — patterns that include partial transcription factor binding
site signatures — in a compact, fixed-length numerical form suitable for
Logistic Regression and tree-based models.  
Sequences containing ambiguous bases (N) are removed before counting.  
**Input:** All six FASTA files (pos/neg × train/val/test)  
**Output:** Six NumPy arrays `pos/neg_train/val/test_X`, each shape (N, 84)


In [15]:
def seq2kmer(in_fa, random_choice=False, rand_n=None, seed=12):
    """
    Compute normalised k-mer frequency matrix (k=1,2,3) from a FASTA file.
    Returns np.ndarray shape (N_clean, 84), dtype float32.
    """
    multi_fa = SeqIO.to_dict(SeqIO.parse(in_fa, "fasta"))
    clean    = {k: v for k, v in multi_fa.items()
                if "N" not in str(v.seq).upper()}

    if random_choice and rand_n is not None:
        random.seed(seed)
        n_available = len(clean)
        if rand_n > n_available:
            print(f"  WARNING: rand_n={rand_n} > available clean sequences "
                  f"({n_available}). Using all {n_available} sequences.")
            rand_n = n_available   # use everything instead of crashing
        keys  = random.sample(list(clean), k=rand_n)
        clean = {k: clean[k] for k in keys}

    nuc  = ["".join(n) for n in itertools.product("ACGT", repeat=1)]
    nuc += ["".join(n) for n in itertools.product("ACGT", repeat=2)]
    nuc += ["".join(n) for n in itertools.product("ACGT", repeat=3)]  # 84 total

    mat = np.zeros((len(clean), len(nuc)), dtype=np.float32)

    for n, (_, record) in enumerate(tqdm(clean.items(),
                                         desc=os.path.basename(in_fa))):
        seq = str(record.seq).upper()
        cnt = {k: 0.0 for k in nuc}
        for i in range(len(seq)):          cnt[seq[i]]     += 1
        for i in range(len(seq) - 1):     cnt[seq[i:i+2]] += 1
        for i in range(len(seq) - 2):     cnt[seq[i:i+3]] += 1
        for k in nuc:
            d = len(seq) - (len(k) - 1)
            cnt[k] /= d if d > 0 else 1
        for j, k in enumerate(nuc):
            mat[n, j] = cnt[k]

    print(f"  {os.path.basename(in_fa)}: {mat.shape}")
    return mat


# ── Positive sets ─────────────────────────────────────────────────────────────
pos_train_X = seq2kmer(r"e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.train.fa")
pos_val_X   = seq2kmer(r"e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.val.fa")
pos_test_X  = seq2kmer(r"e:\3_bio_project_ml\hg19.VISTA.enhancers.chr1.200bp.test.fa")

# ── Negative sets (sample same count as positives for class balance) ───────────
neg_train_X = seq2kmer(r"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.train.clean.fa",
                       random_choice=True, rand_n=pos_train_X.shape[0])
neg_val_X   = seq2kmer(r"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.val.clean.fa",
                       random_choice=True, rand_n=pos_val_X.shape[0])
neg_test_X  = seq2kmer(r"e:\3_bio_project_ml\hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.test.clean.fa",
                       random_choice=True, rand_n=pos_test_X.shape[0])


hg19.VISTA.enhancers.chr1.200bp.train.fa: 100%|██████████| 452417/452417 [01:18<00:00, 5793.94it/s]


  hg19.VISTA.enhancers.chr1.200bp.train.fa: (452417, 84)


hg19.VISTA.enhancers.chr1.200bp.val.fa: 100%|██████████| 129262/129262 [00:23<00:00, 5458.72it/s]


  hg19.VISTA.enhancers.chr1.200bp.val.fa: (129262, 84)


hg19.VISTA.enhancers.chr1.200bp.test.fa: 100%|██████████| 64632/64632 [00:11<00:00, 5395.97it/s]


...

  hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.train.clean.fa: (405057, 84)


hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.val.clean.fa: 100%|██████████| 116267/116267 [00:21<00:00, 5439.24it/s]


  hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.val.clean.fa: (116267, 84)


hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.test.clean.fa: 100%|██████████| 58341/58341 [00:10<00:00, 5451.38it/s]


  hg19.VISTA.non_enhancers.chr1.GCBalanced.200bp.test.clean.fa: (58341, 84)


---
## Step 10 — Create Labels & Merge Datasets
**What we do:** Assign binary labels (1 = enhancer, 0 = non-enhancer) to every
sequence array, then vertically stack positives and negatives for each split.  
**Why it matters:** Supervised learning requires ground-truth labels.
Stacking with matching labels pairs each feature vector with its correct class,
ready for model training.  
**Output:** `train_X / val_X / test_X` (feature matrices) and
`train_y / val_y / test_y` (label vectors)


In [16]:
# Labels: 1 = enhancer, 0 = non-enhancer
pos_train_y = np.ones(pos_train_X.shape[0],  dtype=np.float32)
pos_val_y   = np.ones(pos_val_X.shape[0],    dtype=np.float32)
pos_test_y  = np.ones(pos_test_X.shape[0],   dtype=np.float32)

neg_train_y = np.zeros(neg_train_X.shape[0], dtype=np.float32)
neg_val_y   = np.zeros(neg_val_X.shape[0],   dtype=np.float32)
neg_test_y  = np.zeros(neg_test_X.shape[0],  dtype=np.float32)

# Merge positive + negative for each split
train_X = np.vstack([pos_train_X, neg_train_X])
train_y = np.hstack([pos_train_y, neg_train_y])

val_X   = np.vstack([pos_val_X,   neg_val_X])
val_y   = np.hstack([pos_val_y,   neg_val_y])

test_X  = np.vstack([pos_test_X,  neg_test_X])
test_y  = np.hstack([pos_test_y,  neg_test_y])

print(f"train_X : {train_X.shape}  |  train_y : {train_y.shape}")
print(f"val_X   : {val_X.shape}    |  val_y   : {val_y.shape}")
print(f"test_X  : {test_X.shape}   |  test_y  : {test_y.shape}")
print(f"\nLabel balance (train): "
      f"{int(train_y.sum())} enhancer / {int((train_y == 0).sum())} non-enhancer")


train_X : (857474, 84)  |  train_y : (857474,)
val_X   : (245529, 84)    |  val_y   : (245529,)
test_X  : (122973, 84)   |  test_y  : (122973,)

Label balance (train): 452417 enhancer / 405057 non-enhancer


---
## Step 11 — Save All Features to CSV (86 Columns)
**What we do:** Assemble every sequence's 84 k-mer features, its binary label,
and its dataset split into a single CSV file and save it locally.  
**Why it matters:** A single self-contained CSV decouples feature engineering
from modelling. Any model (LR, RF, SVM, XGBoost, MLP) can be trained by simply
loading this file — no need to re-run the entire pipeline.  
**Column layout (86 total):**

| Group | Columns | Count |
|-------|---------|-------|
| K-mer features (k=1) | `kmer_A`, `kmer_C`, `kmer_G`, `kmer_T` | 4 |
| K-mer features (k=2) | `kmer_AA` … `kmer_TT` | 16 |
| K-mer features (k=3) | `kmer_AAA` … `kmer_TTT` | 64 |
| Label | `label` (1=enhancer, 0=non-enhancer) | 1 |
| Split | `split` (train/val/test) | 1 |
| **Total** | | **86** |

**Output:** `enhancer_kmer_features.csv`


In [ ]:
#  Build column names in the exact order seq2kmer produces them 
nuc  = ["".join(n) for n in itertools.product("ACGT", repeat=1)]
nuc += ["".join(n) for n in itertools.product("ACGT", repeat=2)]
nuc += ["".join(n) for n in itertools.product("ACGT", repeat=3)]
kmer_cols = [f"kmer_{k}" for k in nuc]   # 84 feature column names

#  Stack all splits in order: train then val then test 
X_all = np.vstack([
    pos_train_X, neg_train_X,   # train block (pos first, then neg)
    pos_val_X,   neg_val_X,     # val block
    pos_test_X,  neg_test_X,    # test block
]).astype(np.float32)

y_all = np.hstack([
    pos_train_y, neg_train_y,
    pos_val_y,   neg_val_y,
    pos_test_y,  neg_test_y,
]).astype(np.int8)

split_col = (
    ["train"] * (len(pos_train_y) + len(neg_train_y)) +
    ["val"]   * (len(pos_val_y)   + len(neg_val_y))   +
    ["test"]  * (len(pos_test_y)  + len(neg_test_y))
)

#  Build DataFrame 
df = pd.DataFrame(X_all, columns=kmer_cols)
df["label"] = y_all
df["split"] = split_col

#  Save 
OUTPUT = r"e:\3_bio_project_ml\enhancer_kmer_features.csv"
df.to_csv(OUTPUT, index=False)

print("=" * 55)
print(f"Saved   : {OUTPUT}")
print(f"Shape   : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns : {df.shape[1]} total")
print(f"  - k-mer features : {len(kmer_cols)} (k=1: 4, k=2: 16, k=3: 64)")
print(f"  - label          : 1")
print(f"  - split          : 1")
print("=" * 55)
print(f"\nSplit distribution:")
print(df.groupby(["split", "label"]).size()
        .rename("count")
        .rename_axis(["split", "label (1=enh / 0=non)"])
        .to_string())
print(f"\nFirst 3 rows (first 6 k-mer cols + label + split):")
preview = kmer_cols[:6] + ["label", "split"]
print(df[preview].head(3).to_string(index=False))


Saved   : e:\3_bio_project_ml\enhancer_kmer_features.csv
Shape   : 1225976 rows x 86 columns
Columns : 86 total
  - k-mer features : 84 (k=1: 4, k=2: 16, k=3: 64)
  - label          : 1
  - split          : 1

Split distribution:
split  label (1=enh / 0=non)
test   0                         58341
       1                         64632
train  0                        405057
       1                        452417
val    0                        116267
       1                        129262

First 3 rows (first 6 k-mer cols + label + split):
 kmer_A  kmer_C  kmer_G  kmer_T  kmer_AA  kmer_AC  label split
  0.265   0.195     0.2   0.340 0.095477 0.035176      1 train
  0.265   0.190     0.2   0.345 0.095477 0.035176      1 train
  0.260   0.185     0.2   0.355 0.090452 0.035176      1 train


In [33]:
df=pd.read_csv(r"e:\3_bio_project_ml\enhancer_kmer_features.csv")
df.shape

(1225976, 86)

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

pd.set_option("display.max_colwidth", None)

In [35]:
df=df[:1000]
df.head(10)

,kmer_A,kmer_C,kmer_G,kmer_T,kmer_AA,kmer_AC,kmer_AG,kmer_AT,kmer_CA,kmer_CC,kmer_CG,kmer_CT,kmer_GA,kmer_GC,kmer_GG,kmer_GT,kmer_TA,kmer_TC,kmer_TG,kmer_TT,kmer_AAA,kmer_AAC,kmer_AAG,kmer_AAT,kmer_ACA,kmer_ACC,kmer_ACG,kmer_ACT,kmer_AGA,kmer_AGC,kmer_AGG,kmer_AGT,kmer_ATA,kmer_ATC,kmer_ATG,kmer_ATT,kmer_CAA,kmer_CAC,kmer_CAG,kmer_CAT,kmer_CCA,kmer_CCC,kmer_CCG,kmer_CCT,kmer_CGA,kmer_CGC,kmer_CGG,kmer_CGT,kmer_CTA,kmer_CTC,kmer_CTG,kmer_CTT,kmer_GAA,kmer_GAC,kmer_GAG,kmer_GAT,kmer_GCA,kmer_GCC,kmer_GCG,kmer_GCT,kmer_GGA,kmer_GGC,kmer_GGG,kmer_GGT,kmer_GTA,kmer_GTC,kmer_GTG,kmer_GTT,kmer_TAA,kmer_TAC,kmer_TAG,kmer_TAT,kmer_TCA,kmer_TCC,kmer_TCG,kmer_TCT,kmer_TGA,kmer_TGC,kmer_TGG,kmer_TGT,kmer_TTA,kmer_TTC,kmer_TTG,kmer_TTT,label,split
0,0.265,0.195,0.200,0.340,0.095477,0.035176,0.070352,0.065327,0.050251,0.055276,0.01005,0.080402,0.055276,0.025126,0.055276,0.065327,0.065327,0.075377,0.065327,0.130653,0.040404,0.005051,0.030303,0.020202,0.005051,0.0,0.010101,0.020202,0.015152,0.005051,0.030303,0.020202,0.015152,0.015152,0.005051,0.030303,0.010101,0.015152,0.010101,0.015152,0.025253,0.010101,0.0,0.020202,0.0,0.0,0.0,0.010101,0.015152,0.010101,0.025253,0.030303,0.030303,0.0,0.020202,0.005051,0.010101,0.0,0.0,0.015152,0.015152,0.020202,0.005051,0.015152,0.025253,0.020202,0.015152,0.005051,0.015152,0.015152,0.010101,0.025253,0.010101,0.040404,0.0,0.025253,0.025253,0.0,0.020202,0.020202,0.010101,0.030303,0.020202,0.065657,1,train
1,0.265,0.190,0.200,0.345,0.095477,0.035176,0.070352,0.065327,0.050251,0.050251,0.01005,0.080402,0.055276,0.025126,0.055276,0.065327,0.065327,0.075377,0.065327,0.135678,0.040404,0.005051,0.030303,0.020202,0.005051,0.0,0.010101,0.020202,0.015152,0.005051,0.030303,0.020202,0.015152,0.015152,0.005051,0.030303,0.010101,0.015152,0.010101,0.015152,0.020202,0.010101,0.0,0.020202,0.0,0.0,0.0,0.010101,0.015152,0.010101,0.025253,0.030303,0.030303,0.0,0.020202,0.005051,0.010101,0.0,0.0,0.015152,0.015152,0.020202,0.005051,0.015152,0.025253,0.020202,0.015152,0.005051,0.015152,0.015152,0.010101,0.025253,0.010101,0.040404,0.0,0.025253,0.025253,0.0,0.020202,0.020202,0.010101,0.030303,0.020202,0.070707,1,train
2,0.260,0.185,0.200,0.355,0.090452,0.035176,0.070352,0.065327,0.045226,0.050251,0.01005,0.080402,0.055276,0.025126,0.055276,0.065327,0.065327,0.075377,0.065327,0.145729,0.040404,0.005051,0.030303,0.015152,0.005051,0.0,0.010101,0.020202,0.015152,0.005051,0.030303,0.020202,0.015152,0.015152,0.005051,0.030303,0.005051,0.015152,0.010101,0.015152,0.020202,0.010101,0.0,0.020202,0.0,0.0,0.0,0.010101,0.015152,0.010101,0.025253,0.030303,0.030303,0.0,0.020202,0.005051,0.010101,0.0,0.0,0.015152,0.015152,0.020202,0.005051,0.015152,0.025253,0.020202,0.015152,0.005051,0.015152,0.015152,0.010101,0.025253,0.010101,0.040404,0.0,0.025253,0.025253,0.0,0.020202,0.020202,0.010101,0.030303,0.020202,0.080808,1,train
3,0.260,0.185,0.200,0.355,0.095477,0.035176,0.065327,0.065327,0.045226,0.050251,0.01005,0.080402,0.055276,0.025126,0.055276,0.065327,0.065327,0.075377,0.065327,0.145729,0.040404,0.005051,0.030303,0.020202,0.005051,0.0,0.010101,0.020202,0.015152,0.005051,0.030303,0.015152,0.010101,0.015152,0.005051,0.030303,0.005051,0.015152,0.010101,0.015152,0.020202,0.010101,0.0,0.020202,0.0,0.0,0.0,0.010101,0.015152,0.010101,0.025253,0.030303,0.030303,0.0,0.020202,0.005051,0.010101,0.0,0.0,0.015152,0.015152,0.020202,0.005051,0.015152,0.025253,0.020202,0.015152,0.005051,0.020202,0.015152,0.005051,0.025253,0.010101,0.040404,0.0,0.025253,0.025253,0.0,0.020202,0.020202,0.015152,0.030303,0.020202,0.080808,1,train
4,0.260,0.185,0.200,0.355,0.095477,0.035176,0.065327,0.065327,0.045226,0.050251,0.01005,0.080402,0.055276,0.025126,0.055276,0.060302,0.065327,0.075377,0.070352,0.145729,0.040404,0.005051,0.030303,0.020202,0.005051,0.0,0.010101,0.020202,0.015152,0.005051,0.030303,0.015152,0.010101,0.015152,0.010101,0.030303,0.005051,0.015152,0.010101,0.015152,0.020202,0.010101,0.0,0.020202,0.0,0.0,0.0,0.010101,0.015152,0.010101,0.025253,0.030303,0.030303,0.